Homework 4

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.tsa.api import VAR
import matplotlib.pyplot as plt



In [ ]:
df = pd.read_csv("homework3data.csv")
df.head()

Question 2

In [ ]:
df["rt"] = df.log_excess_ret
df["dy"] = df["div_yld"]
var_df = df[["rt", "dy"]].dropna()
model = VAR(var_df)

bic = []
for lag in [1, 2, 3]:
    VAR_lag = model.fit(lag)
    VAR_bic = VAR_lag.bic
    bic.append(float(VAR_bic))
    print("BIC for lag", lag, ":", VAR_bic)
print("\n", bic)

chosen_lag = bic.index(min(bic)) + 1
print("\nChosen lag:", chosen_lag)

chosen_VAR = model.fit(chosen_lag)
print("\n", chosen_VAR.summary())

Question 3

In [ ]:

def compute_gamma_var (phi1, sigma):

    phi = np.asarray(phi1, float)
    sigma = np.asarray(sigma, float)

    k = phi.shape[0]
    I = np.eye(k * k)
    kronec = np.kron(phi, phi)

    vecSigma = sigma.reshape(-1, order = "F")

    vecGamma = np.linalg.solve(I - kronec, vecSigma)

    gamma = vecGamma.reshape(k, k, order = "F")

    var_vec = np.diag(gamma)

    return gamma , var_vec

phi = chosen_VAR.coefs[0]
sigma = chosen_VAR.sigma_u

gamma, var_vec = compute_gamma_var(phi, sigma)

var_rt = var_vec[0]
var_dy = var_vec[1]
cov_rt_dy = gamma[0][1]

pop_beta = 12 * cov_rt_dy / (var_dy)

pop_beta


Question 4

Estimate a simple restricted VAR. Project the return on just a constant; project the dividend
yield onto a constant and its own lag. Save these coefficients. For each data point t = 2 through
T, you have a vector of residuals. Compute the correlation matrix of these residual vectors. Are
the residuals correlated?


$r_t = \alpha_r + u_{r,t}$ --> Return on just a constant


$dy_t = \alpha_{dy} + \phi  dy_{t-1} + u_{dy,t}$ --> Div Yield on a constant + its own lag

In [ ]:
rt = var_df["rt"].to_numpy()
dy = var_df["dy"].to_numpy()
T  = len(rt)

# 1) r_t on constant
alpha_r = rt.mean()
u_r = rt - alpha_r

# 2) dy_t on constant and dy_{t-1}
X_dy = sm.add_constant(dy[:-1])     # regressors at t=1..T-1
y_dy = dy[1:]                       # dependent at t=2..T
fit_dy = sm.OLS(y_dy, X_dy).fit()
a_dy, phi = fit_dy.params
u_dy = fit_dy.resid                 # length T-1, corresponds to t=2..T

# Residual vectors for t=2..T (with matched indices)
U = np.column_stack([u_r[1:], u_dy])  # shape (T-1, 2)

corr_mat = np.corrcoef(U.T)
corr_mat



Please run a bootstrap under the null of no predictability with these residuals. For each replication,
draw T residual vectors with replacement from the set of observed residual vectors (each observed
vector has probability 1/T ). Reconstruct the return and dividend yield series according to the
restricted VAR and run the regressions in Question 2.1 and 2.3 of Homework 3. Create empirical
distributions of the t-statistics.


In [ ]:
B = 1000
t_q21 = np.empty(B)      # t-stats for HW3 Q2.1
t_q23 = np.empty(B)      # t-stats for HW3 Q2.3
r2_q23 = np.empty(B)     # R^2 for HW3 Q2.3

for b in range(B):
    idx = np.random.randint(0, T-1, size=T-1)  # resample residual vectors
    U_b = U[idx, :]
    e_r = U_b[:, 0]
    e_d = U_b[:, 1]

    # Reconstruct series under restricted VAR (null)
    rt_star = np.empty(T)
    dy_star = np.empty(T)

    dy_star[0] = dy[0]       # initial condition (use observed)
    rt_star[0] = a_r         # not important for t+1 regressions

    for t in range(1, T):
        rt_star[t] = a_r + e_r[t-1]
        dy_star[t] = a_dy + phi * dy_star[t-1] + e_d[t-1]

    # --- HW3 Q2.1 regression (you already have this structure): 12*r_{t+1} on dy_t
    Y1 = 12 * rt_star[1:]
    X1 = sm.add_constant(dy_star[:-1])
    fit1 = sm.OLS(Y1, X1).fit()
    t_q21[b] = fit1.tvalues[1]

    # --- HW3 Q2.3 regression: plug in EXACT same regression as your HW3 code.
    # Placeholder example (replace!):
    Y3 = dy_star[1:]
    X3 = sm.add_constant(dy_star[:-1])
    fit3 = sm.OLS(Y3, X3).fit()
    t_q23[b] = fit3.tvalues[1]
    r2_q23[b] = fit3.rsquared


Do the tests have good size properties for a 5% two-sided test? (It is actually ok to square the
t-statistics and investigate the 5% p-value for a χ2(1).) Also compute the empirical distribution of
the R2 for the regression in Question 2.3 of Homework 3. Describe what you observe. Use 1,000
replications for this bootstrap.

In [ ]:
chi2_95 = 3.841458820694124  # 95% quantile of chi^2(1)
size_q21 = np.mean(t_q21**2 > chi2_95)
size_q23 = np.mean(t_q23**2 > chi2_95)
size_q21, size_q23
